# 02 — Predictive modeling evidence

This notebook is a **read-only review client** for the frozen artifacts produced by `python run_analysis.py`. It does not fit models, tune features, choose calibration, select a threshold, or redesign the analysis after seeing official-test results. The script and modular `src/` package are the sole reproducible analysis path.

The historical circular RPN regression is intentionally absent: it is `HISTORICAL_INVALIDATED_APPROACH`, not a baseline.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


def find_development_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src').is_dir() and (candidate / 'requirements.txt').is_file():
            return candidate
    raise RuntimeError('Open this notebook from inside the repository tree.')


DEV_ROOT = find_development_root()
OUTPUT_ROOT = DEV_ROOT.parent / '05_OUTPUTS'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
SUMMARY_PATH = OUTPUT_ROOT / 'run_summary.json'

required_outputs = [
    TABLE_DIR / 'model_comparison.csv',
    TABLE_DIR / 'ablation_results.csv',
    TABLE_DIR / 'feature_importance.csv',
    TABLE_DIR / 'calibration_development.csv',
    TABLE_DIR / 'calibration_test.csv',
    TABLE_DIR / 'test_predictions.csv',
    TABLE_DIR / 'bootstrap_metrics.csv',
    SUMMARY_PATH,
]
missing = [str(path) for path in required_outputs if not path.is_file()]
if missing:
    raise FileNotFoundError('Run `python run_analysis.py` first. Missing:\n' + '\n'.join(missing))

OUTPUT_ROOT

## Frozen model comparison

Grouped development PR-AUC is the primary selection criterion. Calibration quality and simplicity are supporting criteria, and small score differences are described only as best-performing within this evaluated setup. The required constant-prevalence and age-only baselines answer whether condition signals add value beyond prevalence and simple aging.

In [ ]:
model_comparison = pd.read_csv(TABLE_DIR / 'model_comparison.csv')
model_comparison

## Calibration and threshold discipline

Nested sigmoid calibration was evaluated on grouped development data only and rejected: Brier improvement was approximately 0.000125, below the predefined 0.001 gate, so raw probabilities were retained. The warning threshold is the highest development OOF threshold meeting the predeclared 80% recall target. The official test does not fit the calibrator or choose the threshold. The selected threshold is not an 80% test/field recall guarantee, and the reliability tables do not establish perfect or transferable calibration.

In [ ]:
calibration_development = pd.read_csv(TABLE_DIR / 'calibration_development.csv')
calibration_test = pd.read_csv(TABLE_DIR / 'calibration_test.csv')

display(calibration_development)
display(calibration_test)

## Compact ablations

The ablation study compares `AGE_ONLY`, `CURRENT_SENSORS`, `CURRENT_PLUS_ROLLING`, and `FULL_PARSIMONIOUS_FEATURE_SET` on grouped development evidence. Its purpose is to test whether condition history adds signal—not to search dozens of feature recipes.

In [ ]:
ablation_results = pd.read_csv(TABLE_DIR / 'ablation_results.csv')
ablation_results

## Held-out importance

For the selected scaled Logistic Regression, the final explanation uses absolute standardized coefficients. (The unused Random Forest branch would use post-lock held-out permutation importance.) Both are associative predictive summaries, not causal rankings, sensor-physics claims, or proof that changing a channel changes degradation.

In [ ]:
feature_importance = pd.read_csv(TABLE_DIR / 'feature_importance.csv')
feature_importance.head(15)

## Locked-test artifact integrity

The following cell audits persisted outputs only. It does not refit, recalibrate, move the threshold, or select a different model. Repeatedly modifying the design after reading these artifacts would violate the locked-test policy.

In [ ]:
test_predictions = pd.read_csv(TABLE_DIR / 'test_predictions.csv')
probability_candidates = [
    name for name in ('condition_probability', 'selected_probability', 'probability')
    if name in test_predictions.columns
]
if not probability_candidates:
    raise AssertionError('No documented condition-probability field found.')
probability_column = probability_candidates[0]
probability = pd.to_numeric(test_predictions[probability_column], errors='raise')
assert np.isfinite(probability).all() and probability.between(0.0, 1.0).all()

bootstrap_metrics = pd.read_csv(TABLE_DIR / 'bootstrap_metrics.csv')
pd.DataFrame(
    {
        'artifact': ['test_predictions', 'asset_bootstrap_replicates'],
        'rows': [len(test_predictions), len(bootstrap_metrics)],
    }
)

## Run summary and figure inventory

`run_summary.json` is the machine-readable source for headline values. Generated reports remain the preferred narrative source because they include eligibility rules, policy context, and limitations.

In [ ]:
with SUMMARY_PATH.open('r', encoding='utf-8') as handle:
    run_summary = json.load(handle)

summary_view = pd.json_normalize(run_summary, sep='.').T.rename(columns={0: 'value'})
figure_inventory = pd.DataFrame(
    {'figure': [path.name for path in sorted(FIGURE_DIR.glob('*.png'))]}
)
display(summary_view)
display(figure_inventory)

## Interpretation boundary

The resulting scores can demonstrate predictive signal for the defined near-term event on FD001. PR-AUC—not ROC-AUC restated as accuracy—is the headline. The high ROC-AUC is contextualized by terminal-window labels, correlated rows, age/degradation structure, and official-test truncation. Only 25 test engines are event-eligible, and 16/25 coverage has an approximate Wilson 95% interval of 44.5%–79.8%. These results cannot establish real fleet calibration, production failure prediction, avoided downtime, failure prevention, causal sensor effects, or safety performance.